# Gate 0 (Frame A) — matched-world difference-in-differences, n=10 wikitext-2

**GATE / DIAGNOSTIC — does NOT graduate Phase 3 under any outcome.** Per the precommit
`notes/notes/2026-05-28-gate0-frame-a-valid-control-precommit.md` and the anchor
`notes/notes/2026-05-28-phase3-frame-b-continual-learning-and-gauge-control-finding.md`.

**Question.** Does the *existing* Path C consolidation stack (pull/push + C.2.1–C.2.5)
extract corpus co-occurrence structure, measured against a control that can actually
detect it (a global token-stream shuffle), with the Phase-2 landscape differenced out?

**Arms** (atom seeds 0..9, paired):

| arm | world | consolidation |
|---|---|---|
| A | real | Path C stack |
| B | shuffled-stream | Path C stack |
| C | real | frozen codebook (Phase 2 baseline) |
| D | shuffled-stream | frozen codebook |
| E | real | Path C stack, codebook row-permuted (per-seed gauge confirmation) |

**Primary metric (per-seed DiD, atom seed = unit):**
`DiD_s = [Recall_A(s) − Recall_C(s)] − [Recall_B(s) − Recall_D(s)]` on stratum-pooled
Recall@K. **Pass = mean-DiD 95% CI strictly > 0 AND ≥ 70% of seeds with DiD_s > 0.**
(Per-seed inference — NOT the pseudo-replicated pooled-Wilson gate.)

**Gauge confirmation:** 4a = identity-permutation gauge control byte-identical to A
(checked locally + in-run); 4b = per-seed Recall(gauge control) − Recall(A) CI contains 0.

**Runtime.** Single CUDA process runs 5 arms × 10 seeds + 1 identity cell ≈ 51 cells
sequentially (~30–75 min on A100/L4). Deterministic — if the runtime disconnects, re-run.

**Pre-flight.** Assumes the Gate 0 implementation has been pushed to remote on branch
`codex/phase5-prime-bundle-first-scene-memory`. The verify cell fails fast otherwise.


In [ ]:
# 1. Clone the repo + verify the Gate 0 implementation is present on the branch.
import os
from pathlib import Path
REPO_DIR = '/content/Neuro-AI'
BRANCH = 'codex/phase5-prime-bundle-first-scene-memory'
%cd /content
!rm -rf Neuro-AI
!git clone https://github.com/Dypatterson/Neuro-AI.git
%cd Neuro-AI
!git checkout {BRANCH}
!git log --oneline -3
os.chdir(REPO_DIR)

gate0 = Path(REPO_DIR) / 'experiments' / 'gate0_frame_a.py'
driver = Path(REPO_DIR) / 'experiments' / 'c3_phase3_exit_criterion.py'
if not gate0.exists():
    raise SystemExit('experiments/gate0_frame_a.py not found — push the branch and re-run.')
gsrc = gate0.read_text()
dsrc = driver.read_text()
for needle, where in [
    ('def run_gate0', 'gate0_frame_a.run_gate0'),
    ('def _did_deltas', 'gate0_frame_a DiD helper'),
]:
    if needle not in gsrc:
        raise SystemExit(f'{where} missing — push the branch and re-run.')
for needle, where in [
    ('world: str = "real"', 'c3 driver --world plumbing'),
    ('_collect_per_seed_deltas', 'c3 per-seed CI fix'),
    ('identity_permutation', 'c3 identity-permutation (4a)'),
]:
    if needle not in dsrc:
        raise SystemExit(f'{where} missing — push the branch and re-run.')
print('Gate 0 implementation verified on branch.')


In [ ]:
# 2. Mount Drive for result persistence.
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/neuro-ai/results', exist_ok=True)
print('Drive mounted.')


In [ ]:
# 3. Install deps. wikitext loader uses HF datasets.
!pip install -q torch datasets huggingface_hub


In [ ]:
# 4. Pre-warm the WikiText-2 HF cache (canonical Salesforce/wikitext namespace).
#    Parent CPU-only here; the gate0 run below is a single CUDA process.
import sys; sys.path.insert(0, '/content/Neuro-AI/src')
from datasets import load_dataset
_ = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='train[:1%]')
print('WikiText-2 cache pre-warmed.')


In [ ]:
# 5. GPU info.
!nvidia-smi --query-gpu=name,memory.total,compute_mode --format=csv


In [ ]:
# 6. SMOKE — tiny synthetic Gate 0 on CUDA to confirm runtime + 4a byte-identity.
import subprocess, sys, os, json
from pathlib import Path
os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
smoke_out = Path('reports/gate0_smoke_2026-05-28')
smoke_out.mkdir(parents=True, exist_ok=True)
cmd = [sys.executable, 'experiments/gate0_frame_a.py',
       '--seeds', '0,1,2', '--corpus-source', 'synthetic', '--vocab-size', '40',
       '--D', '256', '--landscape-size', '8', '--window', '4',
       '--n-test-windows', '24', '--n-train-windows', '48',
       '--n-consolidation-events', '50', '--theta-prime-mode', 'default',
       '--device', 'cuda', '--output-dir', str(smoke_out)]
rc = subprocess.call(cmd)
print(f'smoke exit code: {rc}')
if rc != 0:
    raise SystemExit('Gate 0 smoke failed — abort before the full run.')
sm = json.load(open(smoke_out / 'gate0_summary.json'))
assert sm['gauge_confirmation_E']['byte_identical_4a'], '4a byte-identity FAILED in smoke'
print('smoke OK; 4a byte-identical:', sm['gauge_confirmation_E']['byte_identical_4a'],
      '| verdict (synthetic, expect G0->dead):', sm['verdict'])


In [ ]:
# 7. FULL RUN — Gate 0 at the Path C wikitext operating point, n=10, on CUDA.
#    Single process runs all 5 arms × 10 seeds + identity cell sequentially.
import subprocess, sys, os
from pathlib import Path
os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
out_dir = Path('reports/gate0_2026-05-28')
out_dir.mkdir(parents=True, exist_ok=True)
log = Path('reports/gate0_2026-05-28/run.log')
cmd = [sys.executable, 'experiments/gate0_frame_a.py',
       '--seeds', '0,1,2,3,4,5,6,7,8,9',
       '--D', '4096', '--landscape-size', '64', '--window', '8',
       '--n-test-windows', '512', '--n-train-windows', '2048',
       '--K', '5', '--beta', '10', '--n-consolidation-events', '1000',
       '--theta-prime-mode', 'both',
       '--alpha-anti', '0.01', '--repulsion-step-size', '0.05',
       '--lr-pull', '0.1', '--lr-push', '0.05',
       '--corpus-source', 'wikitext', '--wikitext-name', 'wikitext-2-raw-v1',
       '--vocab-cap', '1000',
       '--device', 'cuda', '--output-dir', str(out_dir)]
with log.open('w') as f:
    rc = subprocess.call(cmd, stdout=f, stderr=subprocess.STDOUT)
print(f'gate0 exit code: {rc}')
!tail -60 {log}
if rc != 0:
    raise SystemExit('Gate 0 full run failed — see log above.')


In [ ]:
# 8. Copy results to Drive.
import shutil, os
dst = '/content/drive/MyDrive/neuro-ai/results/gate0_2026-05-28'
os.makedirs(dst, exist_ok=True)
shutil.copytree('reports/gate0_2026-05-28', dst, dirs_exist_ok=True)
print('copied to', dst)
!ls -la {dst}


In [ ]:
# 9. Surface the verdict, DiD, secondaries, and gauge confirmation.
import json
sm = json.load(open('reports/gate0_2026-05-28/gate0_summary.json'))
print('VERDICT:', sm['verdict'])
d = sm['primary_did']['stats']
print(f"\nPrimary DiD: mean={d['mean_delta']:+.4f} "
      f"CI=[{d['ci95_lower']}, {d['ci95_upper']}] "
      f"n={d['n_seeds_used']} | CI>0={d['ci95_above_zero']} "
      f"| >=70%={d['per_seed_robust_ge_threshold']} "
      f"({d['n_seeds_positive']}/{d['n_seeds_used']})")
print('DiD passes (both clauses):', sm['primary_did']['passes'])
ab = sm['secondary']['a_minus_b_whole_pipeline']
ac = sm['secondary']['a_minus_c_consolidation_on_real']
print(f"\n(A)-(B) whole-pipeline: {ab['mean_delta']:+.4f} CI>0={ab['ci95_above_zero']}")
print(f"(A)-(C) cons-on-real:   {ac['mean_delta']:+.4f} CI>0={ac['ci95_above_zero']}")
g = sm['gauge_confirmation_E']
print(f"\nGauge: 4a byte-identical={g['byte_identical_4a']} | "
      f"4b mean={g['mean_delta_4b']:+.4f} CI={g['ci95_4b']} "
      f"contains0={g['ci_contains_zero_4b']} | passes={g['passes_4a_and_4b']}")
print('\n--- full markdown report ---\n')
print(open('reports/gate0_2026-05-28/gate0_summary.md').read())
